In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# Load or create a dummy classifier
def get_classifier():
    # Simulated features: 21 hand landmarks * 3 coords = 63 features
    X = np.random.rand(10, 63)
    y = ['Hello', 'Thanks', 'Hello', 'Thanks', 'Hello', 'Thanks', 'Hello', 'Thanks', 'Hello', 'Thanks']
    clf = KNeighborsClassifier(n_neighbors=3)
    clf.fit(X, y)
    return clf

# Extract 21 landmark x,y,z coords as flat list
def get_hand_features(hand_landmarks):
    return [coord for lm in hand_landmarks.landmark for coord in (lm.x, lm.y, lm.z)]

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
classifier = get_classifier()

cap = cv2.VideoCapture(0)

with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        prediction_text = ''

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                features = get_hand_features(hand_landmarks)
                if len(features) == 63:  # Make sure it’s a full hand
                    features_np = np.array(features).reshape(1, -1)
                    prediction = classifier.predict(features_np)
                    prediction_text = prediction[0]

        # Display result
        cv2.putText(image, f"Prediction: {prediction_text}", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('Sign Language Recognition Demo', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

: 